# Phase 5 on a free GPU (Google Colab)

Runs the **real GPU benchmarks** for the LLM Inference Engine project — no GCP quota needed:
- my engines (naive + batched) under a concurrency sweep
- **vLLM** (PagedAttention) through the same harness via its OpenAI server
- **FP16 vs INT8 vs INT4** quantization study (throughput / GPU memory / quality)

### Before you run
**Runtime → Change runtime type → Hardware accelerator: T4 GPU** → Save. Then *Runtime → Run all*.
At the end it downloads `gpu_results.zip` — send that back and the graphs/dashboard/README get updated.

In [ ]:
!nvidia-smi  # confirm a GPU is attached (T4). If this errors, set the runtime to GPU first.

In [ ]:
# Clone the project
!git clone https://github.com/JATAN2703/llm-inference-engine.git
%cd llm-inference-engine

In [ ]:
# Install deps. vLLM brings a compatible torch+transformers; then our serving/bench deps.
!pip -q install vllm bitsandbytes
!pip -q install fastapi 'uvicorn[standard]' pydantic httpx matplotlib accelerate
import torch; print('CUDA available:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## 1) Benchmark my engines (naive + batched) on the GPU

In [ ]:
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
!python benchmark/runner.py --concurrency 1,4,16,64 --requests 64 --max-tokens 128
!cp results/latest.json results/mine.json
print('saved results/mine.json')

## 2) Benchmark vLLM through the same harness (OpenAI server)

In [ ]:
import subprocess, time, urllib.request
# T4 (compute 7.5) has NO bfloat16 -> force --dtype half. --enforce-eager skips the slow
# CUDA-graph capture / torch.compile step (faster startup, less memory on a 16GB T4).
vllm = subprocess.Popen(
    ['vllm', 'serve', 'Qwen/Qwen2.5-0.5B-Instruct', '--port', '8000',
     '--dtype', 'half', '--enforce-eager',
     '--max-model-len', '2048', '--gpu-memory-utilization', '0.85'],
    stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)
print('waiting for vLLM to load (up to ~8 min)...')
ok = False
for _ in range(96):                                  # 96 * 5s = 8 min
    try:
        if urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2).status == 200:
            ok = True; break
    except Exception:
        pass
    time.sleep(5)
print('vLLM ready' if ok else 'vLLM did NOT come up')
if not ok:                                           # surface the reason so we can fix it
    print('---- last 40 lines of vllm.log ----')
    print(''.join(open('vllm.log').readlines()[-40:]))


In [ ]:
import json, subprocess, urllib.request
# Only benchmark vLLM if it is actually serving -- prevents copying stale (non-vLLM) data.
try:
    healthy = urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3).status == 200
except Exception:
    healthy = False

if healthy:
    subprocess.run(['python', 'benchmark/runner.py', '--url', 'http://127.0.0.1:8000',
                    '--api', 'openai', '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
                    '--engines', 'vllm', '--concurrency', '1,4,16,64',
                    '--requests', '64', '--max-tokens', '128'], check=True)
    rows = json.load(open('results/latest.json'))
    vllm_rows = [r for r in rows if r.get('engine') == 'vllm']   # keep only real vLLM rows
    json.dump(vllm_rows, open('results/vllm.json', 'w'), indent=2)
    print('saved results/vllm.json with', len(vllm_rows), 'rows')
else:
    json.dump([], open('results/vllm.json', 'w'))                # no bogus duplicate data
    print('vLLM not healthy; wrote empty results/vllm.json (skipping vLLM)')


## 3) Quantization study — FP16 vs INT8 vs INT4

In [ ]:
!python benchmark/quant_compare.py fp16,int8,int4

## 4) Merge results, plot, build dashboard, download

In [ ]:
import json
from pathlib import Path
merged = []
for f in ['results/mine.json', 'results/vllm.json']:
    if Path(f).exists():
        merged += json.load(open(f))
json.dump(merged, open('results/latest.json', 'w'), indent=2)
print('merged rows:', len(merged))
!python benchmark/plot.py
!python dashboard/build_dashboard.py

In [ ]:
# package everything to send back
!cd results && zip -qr ../gpu_results.zip . && cd ..
from google.colab import files
files.download('gpu_results.zip')  # then send me this zip